In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test = train_test_split(df,test_size=0.2, random_state=42)

In [ ]:
df.shape

In [ ]:
X_train.shape, X_test.shape

In [ ]:
X_train.dtypes

In [ ]:
df.isna().mean()

In [ ]:
df['Ticket'].value_counts()

In [ ]:
X_train.drop(columns=['Name', 'Cabin', 'Ticket'], inplace=True)
X_test.drop(columns=['Name', 'Cabin', 'Ticket'], inplace=True)

In [ ]:
X_train.head()

In [ ]:
X_train.groupby('Sex').agg({'Survived' : 'mean'})

In [ ]:
X_train['gender_num'] = np.where(X_train['Sex'] == 'female', 1, 0)

In [ ]:
X_test['gender_num'] = np.where(X_test['Sex'] == 'female', 1, 0)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
X_train.groupby('Embarked').agg({'Survived' : 'mean'})

In [ ]:
X_train['embarked_num'] = np.where(X_train['Embarked'] == 'C', 1, 0)
X_test['embarked_num'] = np.where(X_test['Embarked'] == 'C', 1, 0)

In [ ]:
X_train.head()

In [ ]:
X_train.drop(columns=['Sex', 'Embarked'], inplace=True)
X_test.drop(columns=['Sex', 'Embarked'], inplace=True)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
X_train.isna().mean()

In [ ]:
X_train['Age'].mode()

In [ ]:
X_train.fillna(24, inplace=True)

In [ ]:
X_test.fillna(24, inplace=True)

In [ ]:
X_train.isna().mean()

In [ ]:
X_test.isna().mean()

In [ ]:
X_train.head()

In [ ]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

In [ ]:
X_train

In [ ]:
X_test

# Start Training

In [ ]:
y_train = X_train.pop('Survived')
y_test = X_test.pop('Survived')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),  # This will be replaced during grid search
    ('classifier', LogisticRegression(max_iter=1000))
])

scalers = [
    StandardScaler(),
    MinMaxScaler(),
    None  # Option to use no scaler
]

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'scaler': scalers,
    'classifier__C': [0.1, 1, 10],  # Example hyperparameter for Logistic Regression
}

In [ ]:
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=kfold,
    scoring='f1',  # or any other metric
    verbose=2,
    return_train_score=True
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results = results.sort_values('rank_test_score')

In [ ]:
results.columns

In [ ]:
print(results[['param_scaler', 'param_classifier__C', 'mean_train_score', 'mean_test_score']])

In [ ]:
best_pipeline = grid_search.best_estimator_

In [ ]:
!pip install dagshub

In [ ]:
import dagshub

# შემდეგ cell - ში თქვენი რეპოზიტორიის ინფორმაცია შეიყვანეთ

In [ ]:
# import dagshub
# dagshub.init()

In [ ]:
!pip install mlflow

# შემდეგ cell - ში mlflow ზე ვინახავთ მხოლოდ საუკეთესო მოდელს

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Set up the MLflow experiment
mlflow.set_experiment("Logistic_Regression_Seminar")

# Your existing pipeline code
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])
scalers = [
    StandardScaler(),
    MinMaxScaler(),
    None
]

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'scaler': scalers,
    'classifier__C': [0.1, 1, 10],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=kfold,
    scoring='f1',
    verbose=2,
    return_train_score=True
)

# Log each trial in the grid search
with mlflow.start_run(run_name="grid_search_test") as mlflow_run:
    run_id = mlflow_run.info.run_id
    print(run_id)
    
    # Log the parent experiment parameters
    mlflow.log_params({
        "cv_folds": kfold.n_splits,
        "scoring_metric": "f1",
        "random_state": 42,
        "model_type": "LogisticRegression"
    })
    
    # Fit the grid search
    grid_search.fit(X_train, y_train)
    
    # Log best model and parameters as parent run artifacts
    mlflow.log_param("best_params", grid_search.best_params_)
    mlflow.log_metric("best_cv_score", grid_search.best_score_)
    
    # Save best model
    mlflow.sklearn.log_model(grid_search.best_estimator_, "best_model")
    
    # Create a pandas DataFrame of results
    results = pd.DataFrame(grid_search.cv_results_)

# Print out the results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")